In [ ]:
import sys
import threading
import webbrowser
import time

# Make sure the code directory is on the path
sys.path.insert(0, r"C:\Users\domenicbrooks\OneDrive - Research Triangle Institute\CIROH Flash Flooding\code")

# Reload dashboard_mrms in case it was already imported this session
import importlib
import dashboard_mrms
importlib.reload(dashboard_mrms)
from dashboard_mrms import MRMSTransposeDashboard, DEFAULT_NC, DEFAULT_PARQUET

import panel as pn
import holoviews as hv
pn.extension("bokeh")
hv.extension("bokeh")

# ── Configure paths here ──────────────────────────────────────────────────
NC_FILE  = DEFAULT_NC       # ← swap to any other MRMS NetCDF path if needed
SHP_FILE = DEFAULT_PARQUET  # ← set to None to disable watershed overlay
PORT     = 5006
# ─────────────────────────────────────────────────────────────────────────

dashboard = MRMSTransposeDashboard(NC_FILE, shp_path=SHP_FILE)

def _serve():
    pn.serve(dashboard.layout(), port=PORT, show=False, verbose=False)

_server_thread = threading.Thread(target=_serve, daemon=True)
_server_thread.start()

time.sleep(2.0)   # give the server a moment to start (watersheds take a bit longer)

url = f"http://localhost:{PORT}"
print(f"Dashboard running at: {url}")
webbrowser.open(url)


C:\Users\domenicbrooks\OneDrive - Research Triangle Institute\CIROH Flash Flooding\code\dashboard_mrms.py:63: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension("bokeh")


Loading watersheds…
  Loaded 50 watersheds
  Loaded coastline
  Loaded 50 gauge locations


C:\Users\domenicbrooks\OneDrive - Research Triangle Institute\CIROH Flash Flooding\code\dashboard_mrms.py:63: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension("bokeh")


Loading watersheds…
  Loaded 50 watersheds
  Loaded coastline
  Loaded 50 gauge locations


C:\Users\domenicbrooks\AppData\Local\Temp\ipykernel_28492\99657622.py:17: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension("bokeh")


Loading watersheds…
  Loaded 50 watersheds
  Loaded coastline
  Loaded 50 gauge locations
Dashboard running at: http://localhost:5006


True

In [5]:
"""
Sanity check: verify that each calibrated basin maps to the correct
parameter file, and that the values in that file match the best_params CSV
produced by the calibration optimiser.

Also flags any basins where the _cal JSON is missing or whose parameters
deviate unexpectedly from the CSV.
"""
import json
import pandas as pd
from pathlib import Path
from dashboard_mrms import (
    HAWAII_RESULTS_DIR, HYDRO_MODEL_CONFIG_DIR,
    _AVAILABLE_BASIN_IDS, WATERSHED_NAME_COL,
)

# ── 1. Show the name → gauge-ID mapping from the running dashboard ────────
print(f"{'Watershed name':<55} {'Gauge ID'}")
print("-" * 70)
for name, gid in sorted(dashboard._basin_id_for_name.items()):
    print(f"{name:<55} {gid}")

# ── 2. For each basin: load _cal JSON, compare against best_params CSV ────
print("\n\n=== Parameter validation (cal JSON vs. best_params CSV) ===\n")

SAC_KEYS_OPTIMISED = ["uzfwm", "lzfpm", "lzfsm", "lzpk", "lzsk",
                       "adimp", "zperc", "rexp"]   # keys written to best_params CSV

results = []
for basin in _AVAILABLE_BASIN_IDS:
    cal_path = HAWAII_RESULTS_DIR / f"configurations/sac/sac_params_{basin}_cal.json"
    csv_path = HAWAII_RESULTS_DIR / f"output/basin_{basin}/best_params_{basin}_kge_q90.csv"

    if not cal_path.exists():
        results.append({"basin": basin, "status": "MISSING _cal JSON", "max_diff": None})
        continue
    if not csv_path.exists():
        results.append({"basin": basin, "status": "MISSING best_params CSV", "max_diff": None})
        continue

    cal  = json.loads(cal_path.read_text())
    bpdf = pd.read_csv(csv_path).set_index("parameter")["best_value"]

    diffs = {}
    for k in SAC_KEYS_OPTIMISED:
        if k not in bpdf.index:
            continue
        cal_val = cal[k][0] if isinstance(cal[k], list) else cal[k]
        diffs[k] = abs(cal_val - bpdf[k])

    max_diff = max(diffs.values()) if diffs else 0.0
    status = "OK" if max_diff < 1e-4 else f"MISMATCH (max_diff={max_diff:.6f})"
    results.append({"basin": basin, "status": status, "max_diff": max_diff})

summary = pd.DataFrame(results)
pd.set_option("display.max_rows", 60)
print(summary.to_string(index=False))

ok  = (summary["status"] == "OK").sum()
bad = len(summary) - ok
print(f"\n{ok}/{len(summary)} basins passed  |  {bad} issue(s)")


Watershed name                                          Gauge ID
----------------------------------------------------------------------
Alakahi Stream near Kamuela, HI                         16725000
EB of NF Wailua River nr Lihue, Kauai, HI               16068000
Halaulani Str at alt 400 ft nr Kilauea, Kauai, HI       16097500
Halawa Stream near Halawa, Molokai, HI                  16400000
Hanalei River nr Hanalei, Kauai, HI                     16103000
Hanawi Stream near Nahiku, Maui, HI                     16508000
Heeia Stream at Haiku Valley nr Kaneohe, Oahu, HI       16275000
Honokohau Stream near Honokohau, Maui, HI               16620000
Honolii Stream nr Papaikou, HI                          16717000
Honopou Stream near Huelo, Maui, HI                     16587000
Honouliuli Str at H-1 Freeway nr Waipahu, Oahu, HI      16212490
Kahakuloa Stream near Honokohau, Maui, HI               16618000
Kahaluu Str nr Ahuimanu, Oahu, HI                       16283200
Kahana Str at alt 3

In [2]:
import pandas as pd

In [3]:
pd.read_parquet(r"C:\Users\domenicbrooks\Research Triangle Institute\CIROH RTI-UVM-UHM Flash Flood Forecasting Project - General\Modeling\pysac\Hawaii results\usgs_basin_geometry_with_source_all.parquet")

,id,name,source,geometry
0,usgs-01011000,"Allagash River near Allagash, Maine",GAGESII,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00(\x03\x0...
1,usgs-01013500,"Fish River near Fort Kent, Maine",GAGESII,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\xfc\x08...
2,usgs-01015800,"Aroostook River near Masardis, Maine",GAGESII,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00<\x03\x0...
3,usgs-01016500,"MACHIAS RIVER NEAR ASHLAND, ME",GAGESII,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\xbc\x01...
4,usgs-01017000,"Aroostook River at Washburn, Maine",GAGESII,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00=\x04\x0...
...,...,...,...,...
9330,usgs-50141000,"RIO BLANCO NR ADJUNTAS, PR",GAGESII,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\xaf\x00...
9331,usgs-50144000,"RIO GRANDE DE ANASCO NR SAN SEBASTIAN, PR",GAGESII,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\xfc\x02...
9332,usgs-50145395,"RIO CASEI ABV HACIENDA CASEI, PR",GAGESII,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\xa3\x00...
9333,usgs-50147800,"RIO CULEBRINAS AT HWY 404 NR MOCA, PR",GAGESII,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x8f\x02...


In [ ]:
pd.read_parquet(r"C:\Users\domenicbrooks\Research Triangle Institute\CIROH RTI-UVM-UHM Flash Flood Forecasting Project - General\Modeling\pysac\Hawaii results\usgs_lt500_min2014-2024.parquet")

,id,name,state_abbr,is_camels,geometry,begin,end,begin_year,end_year,n_years,area_km2
0,usgs-02408150,"WALNUT CREEK ABOVE CLANTON, ALA",AL,no,b'\x01\x01\x00\x00\x00\x13\x11\x11\x11\x11\xa5...,2014/11/19 06:00:00.000,2026/03/10 04:00:00.000,2014,2026,12,92.5
1,usgs-02423110,"CAHABA RIVER NEAR TRUSSVILLE, ALA",AL,no,b'\x01\x01\x00\x00\x00\xd0F\xc7\xc0\xe6\xa5U\x...,2013/11/08 06:00:00.000,2026/03/10 04:15:00.000,2013,2026,13,32.4
2,usgs-02458190,TRIB TO VILLAGE CREEK AT 50th ST IN BIRMINGHAM,AL,no,b'\x01\x01\x00\x00\x00\xec\x83;l\xba\xb0U\xc0\...,2012/02/15 06:00:00.000,2026/03/10 04:40:00.000,2012,2026,14,8.7
3,usgs-03575890,"PINHOOK CREEK AT CLINTON AVE. AT HUNTSVILLE, AL.",AL,no,b'\x01\x01\x00\x00\x00\xb9\xb9B\x0e\xf9\xa5U\x...,2011/03/09 06:00:00.000,2026/01/09 17:25:00.000,2011,2026,15,58.5
4,usgs-02423160,CAHABA RIVER NEAR WHITES CHAPEL AL,AL,no,b'\x01\x01\x00\x00\x00\x10$NM%\xa3U\xc0\xe6\x0...,2011/08/09 05:00:00.000,2026/03/10 04:15:00.000,2011,2026,15,131.6
...,...,...,...,...,...,...,...,...,...,...,...
3501,usgs-50138000,"RIO GUANAJIBO NR HORMIGUEROS, PR",PR,no,b'\x01\x01\x00\x00\x00\xa3\xc8\xf4\xf5~\xc9P\x...,1988/10/01 04:00:00.000,2026/03/10 07:30:00.000,1988,2026,38,310.8
3502,usgs-50035000,"RIO GRANDE DE MANATI AT CIALES, PR",PR,no,b'\x01\x01\x00\x00\x00_\xa2\xff\x90i\x9dP\xc0\...,1987/02/11 11:15:00.000,2026/03/10 07:30:00.000,1987,2026,39,347.1
3503,usgs-50106100,"RIO COAMO AT HWY 14 AT COAMO, PR",PR,no,b'\x01\x01\x00\x00\x00X\xa4W\t\xaf\x96P\xc0JLr...,1987/05/05 11:15:00.000,2026/03/10 11:00:00.000,1987,2026,39,112.7
3504,usgs-50057000,"RIO GURABO AT GURABO, PR",PR,no,b'\x01\x01\x00\x00\x00\xe0K\xac\x87\xee}P\xc0\...,1985/10/06 04:15:00.000,2026/03/10 11:45:00.000,1985,2026,41,155.9
